In [39]:
import pandas as pd
import re
import os

pd.set_option('display.max_rows', 1000);
pd.set_option('display.max_columns', None);
pd.set_option('future.no_silent_downcasting', True)

In [40]:
os.getcwd()

'/workspaces/no-kill-colorado'

Load all the data

In [29]:
#Links for reference

#Google Sheets 2018-2023
#2023
'https://docs.google.com/spreadsheets/d/1zlh79aLdicsDVjZoxfwVm8SY1iFlKm-GBI34TMk1I30/edit?usp=sharing'
#2022
'https://docs.google.com/spreadsheets/d/1mmsehBjl-V1eVueiCJEjFV6PhM6eKaCCPFIfcrk7IKk/edit?gid=0#gid=0'
#2021
'https://docs.google.com/spreadsheets/d/1E-Suwl9z_e8-W1JuP5HVmIKv6C4mpd73pFpgtCdrTkQ/edit?gid=0#gid=0'
#2020
'https://docs.google.com/spreadsheets/d/1Q5M7Zw_A-Kn2V7csyxGqleVXWliLVQ4e080aNI7oIiE/edit?gid=969721861#gid=969721861'
#2019
'https://docs.google.com/spreadsheets/d/1mY0ckZ7AKwlgeZv5uMA6eah7QwabyPv47SGNuqrO20I/edit?gid=969721861#gid=969721861'
#2018
'https://docs.google.com/spreadsheets/d/1nV6SU6eGIzi0_tz6ccezkEmtJewfioq3/edit?gid=1198915962#gid=1198915962'

#API endpoint 2016-2017
#2017
'https://data.colorado.gov/resource/uhi6-hddy.csv'
#2016
'https://data.colorado.gov/resource/m8vm-brgw.csv'

#Google Sheets 2007-2015
#2015
'https://docs.google.com/spreadsheets/d/1g4MnqPpjTFaYmhjIjeZUuOUdA3mkmvVntfcpD5gtRHY/edit?gid=1203697909#gid=1203697909'
#2014
'https://docs.google.com/spreadsheets/d/1Z6eI4edrGjrb2sJ_4gxYPljB7g60RkBk5BsMxyA_eH4/edit?gid=126874728#gid=126874728'
#2013
    #Intake
'https://docs.google.com/spreadsheets/d/1DjZ9cYKT9sBC1oNBD8HUuS3Zpu1m1qAVETigNPYKNqQ/edit?gid=1450511850#gid=1450511850'
    #Outflow
'https://docs.google.com/spreadsheets/d/1DjZ9cYKT9sBC1oNBD8HUuS3Zpu1m1qAVETigNPYKNqQ/edit?gid=1492755857#gid=1492755857'
#2012
    #Intake
'https://docs.google.com/spreadsheets/d/1cZZFmS-o9QAwi1RrMnoy2gYiF_myL61sr6ynVfNOu6Y/edit?gid=101086594#gid=101086594'
    #Outflow
'https://docs.google.com/spreadsheets/d/1cZZFmS-o9QAwi1RrMnoy2gYiF_myL61sr6ynVfNOu6Y/edit?gid=805617810#gid=805617810'
#2011
    #Intake
'https://docs.google.com/spreadsheets/d/14Y6F_BRnpv4BtMYT4U5QPmCk8rKUB5YF1OCTmpoTkyw/edit?gid=1112172445#gid=1112172445'
    #Outflow
'https://docs.google.com/spreadsheets/d/14Y6F_BRnpv4BtMYT4U5QPmCk8rKUB5YF1OCTmpoTkyw/edit?gid=309603042#gid=309603042'
#2010
    #Intake
'https://docs.google.com/spreadsheets/d/1v0x7Pe_zYEEM0g-MyWXbpC1ZpkqpzCOTuJt3tw-y_XY/edit?gid=1457683636#gid=1457683636'
    #Outflow
'https://docs.google.com/spreadsheets/d/1v0x7Pe_zYEEM0g-MyWXbpC1ZpkqpzCOTuJt3tw-y_XY/edit?gid=1507785229#gid=1507785229'
#2009
    #Intake
'https://docs.google.com/spreadsheets/d/11A67bHzjsJ2SSIyFIoK5MNbIdlSXtORDDtcHA_adcsg/edit?gid=831397415#gid=831397415'
    #Outflow
'https://docs.google.com/spreadsheets/d/11A67bHzjsJ2SSIyFIoK5MNbIdlSXtORDDtcHA_adcsg/edit?gid=765780685#gid=765780685'
#2008
    #Intake
'https://docs.google.com/spreadsheets/d/1Av7-Wr-Lf5G3lbjz6BOJfONgdPqfD_VyeG9eBWazU6A/edit?gid=938588532#gid=938588532'
    #Outflow
'https://docs.google.com/spreadsheets/d/1Av7-Wr-Lf5G3lbjz6BOJfONgdPqfD_VyeG9eBWazU6A/edit?gid=1557235659#gid=1557235659'
#2007
'https://docs.google.com/spreadsheets/d/1CvEIJbREKwJRMFqkTOFWd2u6Bb_JZzhwLkU1JHdiYFE/edit?gid=994387581#gid=994387581'

#Licensing
'https://docs.google.com/spreadsheets/d/1q5PSJaDc-cpfKXBn6tQbg0jVxvwyrCNZUBk-5CCortA/edit?gid=531274251#gid=531274251'

'https://docs.google.com/spreadsheets/d/1q5PSJaDc-cpfKXBn6tQbg0jVxvwyrCNZUBk-5CCortA/edit?gid=531274251#gid=531274251'

In [2]:
# Helper function to convert Google Sheets URL to CSV export URL, skip converting API endpoints
def gsheet_to_csv_url(url):
    if url.endswith('.csv'):
        return url
    if "/edit" in url:
        base = url.split("/edit")[0]
        gid = "0"
        if "gid=" in url:
            gid = url.split("gid=")[-1].split("#")[0]
        return f"{base}/export?format=csv&gid={gid}"
    return url

# List of (year, url) tuples
sheet_links = [
    ("2023", "https://docs.google.com/spreadsheets/d/1zlh79aLdicsDVjZoxfwVm8SY1iFlKm-GBI34TMk1I30/edit?usp=sharing"),
    ("2022", "https://docs.google.com/spreadsheets/d/1mmsehBjl-V1eVueiCJEjFV6PhM6eKaCCPFIfcrk7IKk/edit?gid=0#gid=0"),
    ("2021", "https://docs.google.com/spreadsheets/d/1E-Suwl9z_e8-W1JuP5HVmIKv6C4mpd73pFpgtCdrTkQ/edit?gid=0#gid=0"),
    ("2020", "https://docs.google.com/spreadsheets/d/1Q5M7Zw_A-Kn2V7csyxGqleVXWliLVQ4e080aNI7oIiE/edit?gid=969721861#gid=969721861"),
    ("2019", "https://docs.google.com/spreadsheets/d/1mY0ckZ7AKwlgeZv5uMA6eah7QwabyPv47SGNuqrO20I/edit?gid=969721861#gid=969721861"),
    ("2018", "https://docs.google.com/spreadsheets/d/1nV6SU6eGIzi0_tz6ccezkEmtJewfioq3/edit?gid=1198915962#gid=1198915962"),
    ("2017", "https://data.colorado.gov/resource/uhi6-hddy.csv"), #api endpoint https://dev.socrata.com/foundry/data.colorado.gov/m8vm-brgw
    ("2016", "https://data.colorado.gov/resource/m8vm-brgw.csv"), #api endpoint https://dev.socrata.com/foundry/data.colorado.gov/uhi6-hddy
    ("2015", "https://docs.google.com/spreadsheets/d/1g4MnqPpjTFaYmhjIjeZUuOUdA3mkmvVntfcpD5gtRHY/edit?gid=1203697909#gid=1203697909"),
    ("2014", "https://docs.google.com/spreadsheets/d/1Z6eI4edrGjrb2sJ_4gxYPljB7g60RkBk5BsMxyA_eH4/edit?gid=126874728#gid=126874728"),
    ("2013 Intake", "https://docs.google.com/spreadsheets/d/1DjZ9cYKT9sBC1oNBD8HUuS3Zpu1m1qAVETigNPYKNqQ/edit?gid=1450511850#gid=1450511850"),
    ("2013 Outflow", "https://docs.google.com/spreadsheets/d/1DjZ9cYKT9sBC1oNBD8HUuS3Zpu1m1qAVETigNPYKNqQ/edit?gid=1492755857#gid=1492755857"),
    ("2012 Intake", "https://docs.google.com/spreadsheets/d/1cZZFmS-o9QAwi1RrMnoy2gYiF_myL61sr6ynVfNOu6Y/edit?gid=101086594#gid=101086594"),
    ("2012 Outflow", "https://docs.google.com/spreadsheets/d/1cZZFmS-o9QAwi1RrMnoy2gYiF_myL61sr6ynVfNOu6Y/edit?gid=805617810#gid=805617810"),
    ("2011 Intake", "https://docs.google.com/spreadsheets/d/14Y6F_BRnpv4BtMYT4U5QPmCk8rKUB5YF1OCTmpoTkyw/edit?gid=1112172445#gid=1112172445"),
    ("2011 Outflow", "https://docs.google.com/spreadsheets/d/14Y6F_BRnpv4BtMYT4U5QPmCk8rKUB5YF1OCTmpoTkyw/edit?gid=309603042#gid=309603042"),
    ("2010 Intake", "https://docs.google.com/spreadsheets/d/1v0x7Pe_zYEEM0g-MyWXbpC1ZpkqpzCOTuJt3tw-y_XY/edit?gid=1457683636#gid=1457683636"),
    ("2010 Outflow", "https://docs.google.com/spreadsheets/d/1v0x7Pe_zYEEM0g-MyWXbpC1ZpkqpzCOTuJt3tw-y_XY/edit?gid=1507785229#gid=1507785229"),
    ("2009 Intake", "https://docs.google.com/spreadsheets/d/11A67bHzjsJ2SSIyFIoK5MNbIdlSXtORDDtcHA_adcsg/edit?gid=831397415#gid=831397415"),
    ("2009 Outflow", "https://docs.google.com/spreadsheets/d/11A67bHzjsJ2SSIyFIoK5MNbIdlSXtORDDtcHA_adcsg/edit?gid=765780685#gid=765780685"),
    ("2008 Intake", "https://docs.google.com/spreadsheets/d/1Av7-Wr-Lf5G3lbjz6BOJfONgdPqfD_VyeG9eBWazU6A/edit?gid=938588532#gid=938588532"),
    ("2008 Outflow", "https://docs.google.com/spreadsheets/d/1Av7-Wr-Lf5G3lbjz6BOJfONgdPqfD_VyeG9eBWazU6A/edit?gid=1557235659#gid=1557235659"),
    ("2007", "https://docs.google.com/spreadsheets/d/1CvEIJbREKwJRMFqkTOFWd2u6Bb_JZzhwLkU1JHdiYFE/edit?gid=994387581#gid=994387581"),
    ("Licensing", "https://docs.google.com/spreadsheets/d/1q5PSJaDc-cpfKXBn6tQbg0jVxvwyrCNZUBk-5CCortA/edit?gid=531274251#gid=531274251"),
]

# Download and load each sheet as a DataFrame
dfs = {}
for year, url in sheet_links:
    csv_url = gsheet_to_csv_url(url)
    try:
        dfs[year] = pd.read_csv(csv_url)
        print(f"Loaded {year} data: {dfs[year].shape}")
    except Exception as e:
        print(f"Failed to load {year}: {e}")

# Example: display the first few rows of 2022 data
#dfs["2022"].head()

Loaded 2023 data: (347, 154)
Loaded 2022 data: (369, 154)
Loaded 2021 data: (354, 154)
Loaded 2020 data: (356, 154)
Loaded 2019 data: (349, 154)
Loaded 2018 data: (329, 173)
Loaded 2017 data: (279, 183)
Loaded 2016 data: (260, 204)
Loaded 2015 data: (257, 183)
Loaded 2014 data: (260, 213)
Loaded 2013 Intake data: (282, 47)
Loaded 2013 Outflow data: (282, 59)
Loaded 2012 Intake data: (282, 58)
Loaded 2012 Outflow data: (280, 57)
Loaded 2011 Intake data: (268, 58)
Loaded 2011 Outflow data: (269, 56)
Loaded 2010 Intake data: (264, 55)
Loaded 2010 Outflow data: (265, 57)
Loaded 2009 Intake data: (279, 66)
Loaded 2009 Outflow data: (280, 66)
Loaded 2008 Intake data: (276, 66)
Loaded 2008 Outflow data: (276, 67)
Loaded 2007 data: (297, 27)
Loaded Licensing data: (2977, 7)


Standardize case for names

In [3]:
# Standardize all column names to title case for every DataFrame in dfs

for key in dfs:
    dfs[key].columns = dfs[key].columns.str.title()

In [4]:
# Standardize all string values in facilities_df to title case

def to_title_case(val):
    if isinstance(val, str):
        if val.isupper() or val.islower():
            return val.title()
    return val

for key in dfs:
    dfs[key] = dfs[key].map(to_title_case)

Merge Intake and Outflow for years that have both

In [5]:
# Merge Intake and Outflow datasets on 'pacfa_id' for years that have both

years_with_both = ["2013", "2012", "2011", "2010", "2009", "2008"]
for year in years_with_both:
    intake_key = f"{year} Intake"
    outflow_key = f"{year} Outflow"
    if intake_key in dfs and outflow_key in dfs:
        # Only merge if both have 'Pacfa Id' column
        intake_df = dfs[intake_key]
        outflow_df = dfs[outflow_key]
        if "Pacfa Id" in intake_df.columns and "Pacfa Id" in outflow_df.columns:
            merged = pd.merge(intake_df, outflow_df, on="Pacfa Id", suffixes=("_intake", "_outflow"), how="outer")
            dfs[f"{year}"] = merged
            print(f"Merged {intake_key} and {outflow_key} on Pacfa Id into {year}: {merged.shape}")
        else:
            print(f"Pacfa Id column missing in {intake_key} or {outflow_key}")

Merged 2013 Intake and 2013 Outflow on Pacfa Id into 2013: (625, 105)
Merged 2012 Intake and 2012 Outflow on Pacfa Id into 2012: (283, 114)
Merged 2011 Intake and 2011 Outflow on Pacfa Id into 2011: (293, 113)
Merged 2010 Intake and 2010 Outflow on Pacfa Id into 2010: (270, 111)
Merged 2009 Intake and 2009 Outflow on Pacfa Id into 2009: (315, 131)
Merged 2008 Intake and 2008 Outflow on Pacfa Id into 2008: (276, 132)


Filter Licensing for Shelters, Rescues, and Sanctuaries

In [6]:
# Filter Licensing for organizations classified as Shelters, Rescues, and Sanctuaries

licensing_df = dfs["Licensing"]
mask = licensing_df["Business License App Category Name"].str.contains("Shelter|Rescue|Sanctuary", case=False, na=False)
filtered_licensing = licensing_df[mask]
filtered_licensing.reset_index(inplace=True, drop=True)
print(f"Filtered Licensing: {filtered_licensing.shape}")

dfs["Licensing"] = filtered_licensing

Filtered Licensing: (376, 7)


In [7]:
# Select specific DataFrames from dfs and move them to a new dictionary

selected_keys = ["2023", "2022", "2021", "2020", "2019", "2018", "2017", "2016", "2015", "2014","2013", "2012", "2011", "2010", "2009", "2008", "2007", "Licensing"]
selected_dfs = {k: dfs[k] for k in selected_keys if k in dfs}

# selected_dfs contains only the specified years

Get Facility Location Info

Parse geolocation from existing fields

In [8]:
# --- Split 'location_1' column in 2017 dataset into city, zip code, latitude, longitude ---

def parse_location1_2017(val):
    # Example: 'Denver, CO 80202\n(39.7392, -104.9903)'
    if pd.isnull(val):
        return pd.Series([None, None, None, None])
    parts = str(val).split('\n')
    #print(parts)
    #if len(parts) < 2:
    #    return pd.Series([None, None, None, None])
    city_zip = parts[1]
    #print(city_zip)
    # Second part: '(39.7392, -104.9903)'
    latlon = parts[2]
    # Extract city and zip
    city_zip_match = re.match(r'^(.*),\s*(\d{5})$', city_zip)
    if city_zip_match:
        city = city_zip_match.group(1)
        zip_code = city_zip_match.group(2)
    else:
        city = city_zip
        zip_code = None
    # Extract lat, lon
    latlon_match = re.match(r'^\(([-\d.]+),\s*([-\d.]+)\)', latlon)
    lat = float(latlon_match.group(1)) if latlon_match else None
    lon = float(latlon_match.group(2)) if latlon_match else None
    return pd.Series([city, zip_code, lat, lon])

#transform_2017 = dfs["2017"]

if "2017" in selected_dfs and "Location_1" in selected_dfs["2017"].columns:
    selected_dfs["2017"][["City_From_Loc", "Zip_From_Loc", "Lat", "Lon"]] = selected_dfs["2017"]["Location_1"].apply(parse_location1_2017)

# Show the result
#dfs["2017"][["location_1", "city_from_loc", "zip_from_loc", "lat", "lon"]].head(20)

In [9]:
# --- Split 'location_1' column in 2016 dataset into city, latitude, longitude ---

def parse_location1(val):
    # Example: '\nLongmont, \n(40.165729, -105.101194)'
    if pd.isnull(val):
        return pd.Series([None, None, None])
    # Split on newline
    parts = str(val).split('\n')
    #print(parts)
    city = parts[1].strip(",    ")
    latlon = parts[2]
    # Extract lat, lon
    m2 = re.match(r'^\(([-\d.]+),\s*([-\d.]+)\)', latlon)
    lat = float(m2.group(1)) if m2 else None
    lon = float(m2.group(2)) if m2 else None
    return pd.Series([city, lat, lon])

#transform_2016 = dfs["2016"]

if "2016" in selected_dfs and "Location_1" in selected_dfs["2016"].columns:
    selected_dfs["2016"][["City_From_Loc", "Lat", "Lon"]] = selected_dfs["2016"]["Location_1"].apply(parse_location1)

# Show the result
#dfs["2016"][["location_1", "city_from_loc", "lat", "lon"]].head(20)

In [10]:
# --- Parse 'Location 1' column in 2014 into street address, city, zip, latitude, longitude ---

def parse_location1_2014(val):
    # Example: '123 Main St\nDenver 80202\n(39.7392, -104.9903)'
    if pd.isnull(val):
        return pd.Series([None, None, None, None, None])
    parts = str(val).split('\n')
    #print(parts)
    if len(parts) < 2:
        return pd.Series([None, None, None, None, None])
    # First part: '123 Main St,'
    street = parts[0]
    # Second part: 'Denver 80202'
    city_zip = parts[1]
    # Third part: '(39.7392, -104.9903)'
    latlon_part = parts[2]
    # Extract city and zip
    city_zip_match = re.match(r'^(.*)\s*(\d{5})$', city_zip)
    if city_zip_match:
        city = city_zip_match.group(1)
        zip_code = city_zip_match.group(2)
    else:
        city = city_zip
        zip_code = None
    # Extract lat, lon
    latlon_match = re.match(r'^\(([-\d.]+),\s*([-\d.]+)\)', latlon_part)
    lat = float(latlon_match.group(1)) if latlon_match else None
    lon = float(latlon_match.group(2)) if latlon_match else None
    return pd.Series([street, city, zip_code, lat, lon])

#transform_2014 = dfs["2014"]

if "2014" in selected_dfs and "Location 1" in selected_dfs["2014"].columns:
    selected_dfs["2014"][["Street_Address", "City_From_Loc", "Zip_From_Loc", "Lat", "Lon"]] = selected_dfs["2014"]["Location 1"].apply(parse_location1_2014)

# Show the result
#dfs["2014"][["Location 1", "street_address", "city_from_loc", "zip_from_loc", "lat", "lon"]].head(10)

Combine location fields into single dataframe

In [60]:
# Mapping of possible column names for each field
pacfa_id_cols = ['Pacfa Id'
                 ]
license_no_cols = ['Pacfa_License_Number',
                   'Pacfa License Number:'
                   ]
business_type_cols = ['Business License App Category Name'
                      ]
facility_cols = ['Account Name',
                 'DBA',
                 'Facility Name',
                 'Facility_Name', 
                 'Facility Name:',
                 'Name_intake',
                 'Name_outflow',
                 'Name'
                 ]
address_cols = ['Facility Physical Street Address',
                'Facility Street Address',
                'Street_Address',
                'Address_intake',
                'Address_outflow',
                'Address'
                ]
city_cols = ['City', 
             'City_From_Loc',
             'City_intake',
             'City_outflow',
             ]
state_cols = ['State', 
              'State_intake',
              'State_outflow',
              ]
zip_cols = ['Zip_Code',
            'Zip_From_Loc',
            'Zip Code',
            'Zip_intake',
            'Zip_outflow',
            'Zip'
            ]
county_cols = ['County'
               ]
lat_cols = ['Lat']
long_cols = ['Lon']

def find_column(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

records = []
for year, df in selected_dfs.items():
    cols = df.columns
    pcol = find_column(cols, pacfa_id_cols)
    lcol = find_column(cols, license_no_cols)
    bcol = find_column(cols, business_type_cols)
    fcol = find_column(cols, facility_cols)
    acol = find_column(cols, address_cols)
    ccol = find_column(cols, city_cols)
    scol = find_column(cols, state_cols)
    zcol = find_column(cols, zip_cols)
    kcol = find_column(cols, county_cols)
    latcol = find_column(cols, lat_cols)
    longcol = find_column(cols, long_cols)

    if fcol is None:
        continue
    sub = pd.DataFrame()
    sub['year'] = [year] * len(df)
    sub['pacfa_id'] = df[pcol] if pcol else None
    sub['license_no'] = df[lcol] if lcol else None
    sub['business_type'] = df[bcol] if bcol else None
    sub['facility_name'] = df[fcol]
    sub['address'] = df[acol] if acol else None
    sub['city'] = df[ccol] if ccol else None
    sub['state'] = df[scol] if scol else None
    sub['zip_code'] = df[zcol] if zcol else None
    sub['county'] = df[kcol] if kcol else None
    sub['latitude'] = df[latcol] if latcol else None
    sub['longitude'] = df[longcol] if longcol else None
    records.append(sub)

facilities_df = pd.concat(records, ignore_index=True)
facilities_df = facilities_df.drop_duplicates().dropna(subset=['facility_name'])
facilities_df.reset_index(drop=True, inplace=True)


/tmp/ipykernel_54054/2953905490.py:84: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  facilities_df = pd.concat(records, ignore_index=True)


To Do: Assign ID to Facility Name; Map name variations to same ID

Fill missing locations with adjacent info (if to do is done, group by IDs instead)

In [61]:
# Group by facility_name and fill empty values with the most recent (last valid) value

# First, sort by facility_name and year (most recent last)
facilities_df = facilities_df.sort_values(by=["facility_name", "year"])

# Remove values from license_no that don't start with 'Pl' or 'Ag', but keep null values
facilities_df['license_no'] = facilities_df['license_no'].where(
    facilities_df['license_no'].isna() | facilities_df['license_no'].str.startswith(('Pl', 'Ag')),
    None
)

# Fill missing values within each facility_name group, using the most recent (last valid) value
facilities_df = facilities_df.groupby("facility_name")[["year","pacfa_id","license_no","business_type","facility_name","address","city","state","zip_code","county","latitude","longitude"]].apply(lambda g: g.ffill().bfill(), include_groups=True)

# Reset index after groupby
facilities_df = facilities_df.reset_index(drop=True)

In [62]:
facilities_df.head(50)

,year,pacfa_id,license_no,business_type,facility_name,address,city,state,zip_code,county,latitude,longitude
0,2014,None,None,None,104603,NaN,None,None,None,NaN,NaN,NaN
1,2015,None,Pl002Ba1,None,2 Blondes All Breed Rescue,7695 Louviers Blvd,Littleton,None,80131,Douglas County,39.612653,-105.016198
2,2016,None,Pl002Ba1,None,2 Blondes All Breed Rescue,7695 Louviers Blvd,Littleton,None,80126,Douglas County,39.612653,-105.016198
3,2017,None,Pl002Ba1,None,2 Blondes All Breed Rescue,7695 Louviers Blvd,Littleton,None,80126,Douglas County,39.543528,-104.960930
4,2019,None,Pl002Ba1,None,2 Blondes All Breed Rescue,7695 Louviers Blvd,Littleton,None,80126,Douglas County,39.543528,-104.960930
5,2020,None,None,Pet Animal Large Rescue,"2 Blondes All Breed Rescue, Inc.",None,Lakewood,Co,None,Jefferson,NaN,NaN
6,2021,None,None,Pet Animal Large Rescue,"2 Blondes All Breed Rescue, Inc.",None,Lakewood,Co,None,Jefferson,NaN,NaN
7,2022,None,None,Pet Animal Large Rescue,"2 Blondes All Breed Rescue, Inc.",None,Lakewood,Co,None,Jefferson,NaN,NaN
8,2023,None,None,Pet Animal Large Rescue,"2 Blondes All Breed Rescue, Inc.",None,Lakewood,Co,None,Jefferson,NaN,NaN
9,Licensing,None,None,Pet Animal Large Rescue,"2 Blondes All Breed Rescue, Inc.",None,Lakewood,Co,None,Jefferson,NaN,NaN


In [63]:
facilities_df.to_csv("/workspaces/no-kill-colorado/facility_location.csv",index=False)

Combine Measures

In [31]:
# Build a dataframe of year, facility_name, animal_type, measure, value for all selected_dfs, combining all non-dog/cat types as "Other"

animal_types = ['Dog', 'Cat', 'Rabbit', 'Bird', 'Small Mammal', 'Reptile', 'Amphibian', 'Other']
measure_keywords = {
    'Beginning Inventory': ['1/1/','beginning count','beginning foster count','beg. inven.','beginning inventory'],
    'Ending Inventory': ['12/31/','ending count','foster count','ending inventory'],

    'Stray': ['stray'],
    'Owner Relinquished': ['owner relinquished','owner surr.','owners surrendered','returned'],
    'Transfer In': ['transfer from','transfer in','transfer within','trans. in from in state','trans. in from out of state','trans in state','trans out of state'],
    'Other Intake': ['other','other:','tnr','protective','disaster','confiscate','confiscated','protec. cust.','protected','accepted'],

    'Adoption': ['adoption','adopted'],
    'Returned': ['returned to owner','rto','return to owner'],
    'Transfer Out': ['transfer to', 'transfer out','transferred out','transferred to','rescue out','trans. in state','trans. out of state','trans in state','trans out of state','transferred/facilities'],
    'Other Outflow': ['other.1','other live','other: live','other outcomes'],

    'Death': ['deaths','died'],
    'Euthanasia': ['euthanasia','euthanized','doa'], #would include ORE for years that have it
    'Missing': ['missing', 'stolen'],
}

def extract_animal_measure(col):
    found_animal = None
    for animal in animal_types:
        if animal.lower() in col.lower():
            found_animal = animal
            break
    # If not Dog or Cat, group as "Other"
    if found_animal not in ['Dog', 'Cat']:
        found_animal = 'Other'
    for canonical, keywords in measure_keywords.items():
        for kw in keywords:
            if kw in col.lower():
                return found_animal, canonical
    return None, None

records = []
for year, df in selected_dfs.items():
    # Try to find a facility name column
    facility_col = None
    for c in ['Account Name',
                 'DBA',
                 'Facility Name',
                 'Facility_Name', 
                 'Facility Name:',
                 'Name_intake',
                 'Name_outflow',
                 'Name']:
        if c in df.columns:
            facility_col = c
            break
    for col in df.columns:
        animal, measure = extract_animal_measure(col)
        if animal and measure:
            for idx, val in df[col].items():
                facility_name = df[facility_col].iloc[idx] if facility_col else None
                records.append({
                    'year': year,
                    'facility_name': facility_name,
                    'animal_type': animal,
                    'measure': measure,
                    'value': val
                })

animal_measure_df = pd.DataFrame(records)
animal_measure_df = animal_measure_df.dropna(subset=['value'])
#animal_measure_df.head()

In [ ]:
# Specify data types for each column of animal_measure_df
animal_measure_df = animal_measure_df.astype({
    'year': 'string',
    'facility_name': 'string',
    'animal_type': 'string',
    'measure': 'string',
    'value': 'string'
})
animal_measure_df['value'] = pd.to_numeric(animal_measure_df['value'], errors='coerce').fillna(0).astype(int)

In [64]:
# Group animal_measure_df by year, facility_name, animal_type, measure and sum values

animal_measure_grouped = (
    animal_measure_df.groupby(['year', 'facility_name', 'animal_type', 'measure'], as_index=False).agg({'value': 'sum'})
)

animal_measure_grouped.head(50)

,year,facility_name,animal_type,measure,value
0,2007,4 Bar D Kennels,Cat,Adoption,2
1,2007,4 Bar D Kennels,Cat,Other Intake,4
2,2007,4 Bar D Kennels,Dog,Adoption,36
3,2007,4 Bar D Kennels,Dog,Euthanasia,2
4,2007,4 Bar D Kennels,Dog,Other Intake,38
5,2007,4 Paws Rescue,Dog,Adoption,113
6,2007,4 Paws Rescue,Dog,Other Intake,118
7,2007,9 Lives Rescue,Cat,Adoption,511
8,2007,9 Lives Rescue,Cat,Death,23
9,2007,9 Lives Rescue,Cat,Euthanasia,10


In [65]:
animal_measure_grouped.to_csv("/workspaces/no-kill-colorado/facility_measure.csv",index=False)

In [66]:
# Merge facilities_df and animal_measure_grouped on year and facility_name
merged_df = pd.merge(
    facilities_df,
    animal_measure_grouped,
    on=['year', 'facility_name'],
    how='right'
)

# Optionally, save or display the merged result
merged_df.head(100)

,year,pacfa_id,license_no,business_type,facility_name,address,city,state,zip_code,county,latitude,longitude,animal_type,measure,value
0,2007,4270.0,None,None,4 Bar D Kennels,40733 Cr 46,Fleming,Co,80728.0,None,NaN,NaN,Cat,Adoption,2
1,2007,4270.0,None,None,4 Bar D Kennels,40733 Cr 46,Fleming,Co,80728.0,None,NaN,NaN,Cat,Other Intake,4
2,2007,4270.0,None,None,4 Bar D Kennels,40733 Cr 46,Fleming,Co,80728.0,None,NaN,NaN,Dog,Adoption,36
3,2007,4270.0,None,None,4 Bar D Kennels,40733 Cr 46,Fleming,Co,80728.0,None,NaN,NaN,Dog,Euthanasia,2
4,2007,4270.0,None,None,4 Bar D Kennels,40733 Cr 46,Fleming,Co,80728.0,None,NaN,NaN,Dog,Other Intake,38
5,2007,4636.0,Pl000Lmt,None,4 Paws Rescue,1332 Imperial Dr.,Colorado Springs,Co,80918.0,El Paso County,NaN,NaN,Dog,Adoption,113
6,2007,4636.0,Pl000Lmt,None,4 Paws Rescue,1332 Imperial Dr.,Colorado Springs,Co,80918.0,El Paso County,NaN,NaN,Dog,Other Intake,118
7,2007,2540.0,Pl0009H9,Pet Animal Small Rescue,9 Lives Rescue,2159 Brookwood Dr.,Colorado Springs,Co,80918.0,El Paso County,38.874109,-104.719462,Cat,Adoption,511
8,2007,2540.0,Pl0009H9,Pet Animal Small Rescue,9 Lives Rescue,2159 Brookwood Dr.,Colorado Springs,Co,80918.0,El Paso County,38.874109,-104.719462,Cat,Death,23
9,2007,2540.0,Pl0009H9,Pet Animal Small Rescue,9 Lives Rescue,2159 Brookwood Dr.,Colorado Springs,Co,80918.0,El Paso County,38.874109,-104.719462,Cat,Euthanasia,10


In [67]:
merged_df.to_csv("/workspaces/no-kill-colorado/facility_location_measure_combined.csv",index=False)

QA